# Preprocess single-cell RNA and ATAC after Cell Ranger

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manarai/Bio559R-studio1/blob/main/tutorials/saver_scanpy/preprocess_saver_scanpy_rna_atac.ipynb)

**BIO559R student tutorial — Scanpy + Muon with optional SAVER RNA recovery**

This notebook starts with a **filtered 10x count matrix** produced by Cell Ranger, Cell Ranger ATAC, or Cell Ranger ARC. It is an exploratory preprocessing workflow, not an automated source of final cell identities or biological claims.

> **Critical scope:** SAVER is an optional R method for post-QC **RNA UMI counts only**. It does not process ATAC peaks. This notebook preserves raw RNA counts in `rna.layers["counts"]` and never overwrites them with SAVER estimates.

## Learning checkpoints

1. Define your input route and data-access constraints.
2. Inspect and filter RNA and/or ATAC QC metrics with a written rationale.
3. Build an exploratory graph, UMAP, and clusters for each modality.
4. Optionally export post-QC RNA counts to the SAVER R runner.
5. Save reproducible objects and an analysis log.


## Before running anything

- Work from an approved project directory. Do **not** commit data or results to this teaching repository.
- Record species/reference, assay, chemistry, Cell Ranger/ARC version, sample and batch labels, and the biological question.
- Use approved storage and data-sharing practices. Do not upload restricted human data to Colab or a public AI tool.
- Do not use universal filtering thresholds. Plot first; then choose and document starting values appropriate to the study.

**AI checkpoint A — ask an AI assistant to explain, not decide:**

> “I have a [RNA-only / ATAC-only / multiome] 10x dataset from [species/reference] with [N] samples and [batches]. Explain the needed input files and QC metrics. Ask clarifying questions before proposing *starting* thresholds. Do not run commands, modify files, or make biological claims.”


In [ ]:
# Colab setup: run this cell only in Google Colab.
# For local/HPC work, use the Conda environment in environment.yml instead.
# Colab storage is ephemeral: use only small, approved, non-sensitive teaching data.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install "scanpy>=1.11,<1.13" "muon>=0.1.9,<0.2" leidenalg igraph scikit-misc pysam pyranges
    print("Python single-cell packages installed for this Colab runtime.")
    print("SAVER is not installed here; use the reproducible Conda environment for the optional R SAVER stage.")
else:
    print("Not in Colab. Activate the 'saver-scanpy' Conda environment before continuing.")


In [ ]:
# Imports and plotting defaults
from pathlib import Path
import json
import platform
import sys

import anndata as ad
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import io

sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, facecolor="white")
print("Python:", sys.version.split()[0])
print("Scanpy:", sc.__version__)
print("Muon:", mu.__version__)


## 1. Data contract: choose one input route

Place paths below in an approved **analysis project**, not in the Git repository. The HDF5 file is typically inside a Cell Ranger / ARC `outs/` directory.

| Route | HDF5 input | Optional companion file | Object used below |
| --- | --- | --- | --- |
| RNA-only | `filtered_feature_bc_matrix.h5` | sample metadata table | `AnnData` (`rna`) |
| ATAC-only | `filtered_peak_bc_matrix.h5` | `atac_fragments.tsv.gz` | `AnnData` (`atac`) |
| Multiome | `filtered_feature_bc_matrix.h5` | `atac_fragments.tsv.gz` | `MuData` (`mdata`) |

For ATAC fragment-based QC, the fragments file and peak coordinates must match the genome/reference. Multiome matrices are read by Muon and split into RNA and ATAC modalities. Set unneeded paths to `None`.


In [ ]:
# EDIT ONLY THESE PATHS FOR YOUR SELECTED ROUTE.
# Example assumes files were staged under an approved project directory.
PROJECT = Path("/path/to/approved/single_cell_project")
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RNA_H5 = PROJECT / "data/rna/filtered_feature_bc_matrix.h5"
ATAC_H5 = PROJECT / "data/atac/filtered_peak_bc_matrix.h5"
ATAC_FRAGMENTS = PROJECT / "data/atac/atac_fragments.tsv.gz"
MULTIOME_H5 = PROJECT / "data/multiome/filtered_feature_bc_matrix.h5"
MULTIOME_FRAGMENTS = PROJECT / "data/multiome/atac_fragments.tsv.gz"

# Choose exactly one: "rna", "atac", or "multiome".
INPUT_ROUTE = "rna"
assert INPUT_ROUTE in {"rna", "atac", "multiome"}

required = {"rna": RNA_H5, "atac": ATAC_H5, "multiome": MULTIOME_H5}[INPUT_ROUTE]
if not required.exists():
    raise FileNotFoundError(
        f"Selected route '{INPUT_ROUTE}' requires {required}. Edit the paths above; do not continue with a guessed file."
    )
print("Selected route:", INPUT_ROUTE)
print("Input:", required)


In [ ]:
# Load the selected route. Do not filter before inspecting the raw matrix.
rna = atac = mdata = None

if INPUT_ROUTE == "rna":
    rna = sc.read_10x_h5(RNA_H5, gex_only=True)
    rna.var_names_make_unique()
    rna.layers["counts"] = rna.X.copy()  # invariant: unnormalized counts
    print(rna)

elif INPUT_ROUTE == "atac":
    mdata = mu.read_10x_h5(ATAC_H5)
    if "atac" not in mdata.mod:
        raise ValueError(f"Expected an 'atac' modality, found {list(mdata.mod)}. Confirm this is a 10x ATAC peak matrix.")
    atac = mdata.mod["atac"]
    atac.var_names_make_unique()
    atac.layers["counts"] = atac.X.copy()
    if ATAC_FRAGMENTS.exists():
        atac.uns.setdefault("files", {})["fragments"] = str(ATAC_FRAGMENTS)
    print(atac)

else:
    mdata = mu.read_10x_h5(MULTIOME_H5)
    mdata.var_names_make_unique()
    if "rna" not in mdata.mod or "atac" not in mdata.mod:
        raise ValueError("Expected both 'rna' and 'atac' modalities. Confirm that this is Cell Ranger ARC multiome output.")
    rna, atac = mdata.mod["rna"], mdata.mod["atac"]
    rna.layers["counts"] = rna.X.copy()
    atac.layers["counts"] = atac.X.copy()
    if MULTIOME_FRAGMENTS.exists():
        atac.uns.setdefault("files", {})["fragments"] = str(MULTIOME_FRAGMENTS)
    print(mdata)


## 2. RNA quality control and exploratory analysis

The following is a **starting workflow**, not a universal filter. Check distributions by sample/batch; a high mitochondrial fraction, library size, or detected-gene count can have biological as well as technical causes. Assess doublets with a method appropriate to your experimental design rather than deleting cells automatically.

**AI checkpoint B:** show an assistant the *summarized/approved* QC distributions and ask it to explain the tradeoffs of several proposed cutoffs. Then choose the rule yourself and record why it is appropriate for your samples.


In [ ]:
# RNA QC metrics and diagnostic plots — runs only for RNA-containing routes.
if rna is not None:
    # Supports common human (MT-) and mouse (mt-) gene naming conventions.
    rna.var["mt"] = rna.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
    display(rna.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())
    sc.pl.violin(rna, ["total_counts", "n_genes_by_counts", "pct_counts_mt"], jitter=0.25, multi_panel=True)
    sc.pl.scatter(rna, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")
else:
    print("Skipping RNA QC: this is an ATAC-only route.")


In [ ]:
# Choose documented *starting* RNA QC rules after reviewing the preceding plots.
# Replace these values with cohort-appropriate choices; record the rationale in a methods log.
RNA_MIN_GENES = 200
RNA_MAX_GENES = None          # e.g., set after assessing high-complexity/doublet candidates
RNA_MAX_PCT_MT = None         # e.g., set after inspecting tissue-specific distributions
RNA_MIN_CELLS_PER_GENE = 3

if rna is not None:
    before = rna.n_obs
    keep = rna.obs["n_genes_by_counts"] >= RNA_MIN_GENES
    if RNA_MAX_GENES is not None:
        keep &= rna.obs["n_genes_by_counts"] <= RNA_MAX_GENES
    if RNA_MAX_PCT_MT is not None:
        keep &= rna.obs["pct_counts_mt"] <= RNA_MAX_PCT_MT
    rna = rna[keep].copy()
    # Keep the original count layer when subsetting; never regenerate it from normalized values.
    sc.pp.filter_genes(rna, min_cells=RNA_MIN_CELLS_PER_GENE)
    print(f"RNA cells retained: {rna.n_obs:,} / {before:,}; genes retained: {rna.n_vars:,}")
    print("Counts layer retained:", "counts" in rna.layers)


**AI checkpoint C:** Before running PCA or clustering, ask an AI assistant to audit whether the raw-count layer was preserved, whether a batch covariate is present, and which graph parameters merit a sensitivity check. It should not choose a cluster resolution or cell identity for you.


In [ ]:
# RNA: normalize a working matrix, then build an exploratory graph.
# The raw UMI counts remain in rna.layers['counts'].
if rna is not None:
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, layer="counts", flavor="seurat_v3", n_top_genes=2000)
    sc.pl.highly_variable_genes(rna)
    # Preserve log-normalized expression for marker ranking before scaling PCA inputs.
    rna.raw = rna

    # Scaling is only for PCA/graph construction; it does not replace the counts layer.
    sc.pp.scale(rna, max_value=10)
    sc.tl.pca(rna, use_highly_variable=True, svd_solver="arpack")
    sc.pp.neighbors(rna, n_neighbors=15, n_pcs=30)
    sc.tl.umap(rna, random_state=0)
    sc.tl.leiden(rna, resolution=0.5, key_added="leiden_rna")
    sc.pl.umap(rna, color=["leiden_rna", "total_counts", "pct_counts_mt"])

    sc.tl.rank_genes_groups(rna, groupby="leiden_rna", method="wilcoxon", use_raw=True)
    sc.pl.rank_genes_groups(rna, n_genes=15, sharey=False)


## 3. Optional SAVER: export post-QC RNA UMI counts

Run this section **only if RNA has passed your QC review** and you have a specific reason to assess expression recovery. SAVER expects **genes × cells**, unnormalized counts. The exported files are sparse; the R runner creates an `.rds` object containing posterior estimates and standard errors.

- SAVER can be computationally intensive. Start with a pilot subset and a small core count.
- Do not run SAVER on ATAC peaks or log-normalized RNA values.
- Do not silently replace raw counts or use recovered values as the default for differential expression.

**AI checkpoint D:** ask an assistant to estimate computational implications and review the command in the README, but execute it yourself on approved infrastructure.


In [ ]:
# Export post-QC RNA counts in the orientation required by SAVER: genes x cells.
if rna is not None:
    SAVER_INPUT = RESULTS / "saver_input"
    SAVER_INPUT.mkdir(parents=True, exist_ok=True)
    counts_genes_by_cells = rna.layers["counts"].T
    io.mmwrite(SAVER_INPUT / "rna_counts_genes_by_cells.mtx", counts_genes_by_cells)
    pd.Series(rna.var_names).to_csv(SAVER_INPUT / "genes.tsv", sep="\t", index=False, header=False)
    pd.Series(rna.obs_names).to_csv(SAVER_INPUT / "barcodes.tsv", sep="\t", index=False, header=False)
    manifest = {
        "shape_genes_by_cells": list(counts_genes_by_cells.shape),
        "source_layer": "rna.layers['counts']",
        "note": "Post-QC unnormalized RNA counts; not ATAC, not log-normalized values.",
    }
    (SAVER_INPUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Wrote SAVER input to", SAVER_INPUT)
    print("Next: run tutorials/saver_scanpy/run_saver.R from a terminal after reviewing the README.")


## 4. ATAC quality control and LSI workflow

ATAC requires its own QC logic. Counts/peaks are **not** treated as RNA genes. Fragment-based metrics (nucleosome signal and TSS enrichment) require an accessible, compatible fragments file; they are intentionally not run automatically below because they can be expensive and depend on annotation/reference consistency.

**AI checkpoint E:** ask an assistant to explain the QC distribution and data requirements, but do not accept arbitrary TSS/nucleosome cutoffs without considering your assay and tissue.


In [ ]:
# ATAC QC metrics and basic distributions.
if atac is not None:
    sc.pp.calculate_qc_metrics(atac, percent_top=None, log1p=False, inplace=True)
    display(atac.obs[["total_counts", "n_genes_by_counts"]].describe())
    sc.pl.violin(atac, ["total_counts", "n_genes_by_counts"], jitter=0.25, multi_panel=True)
    sc.pl.scatter(atac, x="total_counts", y="n_genes_by_counts")
    fragment_path = atac.uns.get("files", {}).get("fragments")
    print("Fragments registered for optional fragment-based QC:", fragment_path or "No")
else:
    print("Skipping ATAC QC: this is an RNA-only route.")


In [ ]:
# ATAC filter choices are placeholders. Choose after inspecting cohort-specific distributions.
ATAC_MIN_PEAKS = 2000
ATAC_MAX_PEAKS = None
ATAC_MIN_COUNTS = None
ATAC_MIN_CELLS_PER_PEAK = 10

if atac is not None:
    before = atac.n_obs
    keep = atac.obs["n_genes_by_counts"] >= ATAC_MIN_PEAKS
    if ATAC_MAX_PEAKS is not None:
        keep &= atac.obs["n_genes_by_counts"] <= ATAC_MAX_PEAKS
    if ATAC_MIN_COUNTS is not None:
        keep &= atac.obs["total_counts"] >= ATAC_MIN_COUNTS
    atac = atac[keep].copy()
    sc.pp.filter_genes(atac, min_cells=ATAC_MIN_CELLS_PER_PEAK)
    print(f"ATAC cells retained: {atac.n_obs:,} / {before:,}; peaks retained: {atac.n_vars:,}")

    # TF-IDF -> LSI -> graph -> UMAP -> Leiden.
    # Inspect the first LSI dimension before excluding it; it often reflects depth.
    mu.atac.pp.tfidf(atac, scale_factor=1e4)
    mu.atac.tl.lsi(atac)
    atac.obsm["X_lsi_no1"] = atac.obsm["X_lsi"][:, 1:]
    sc.pp.neighbors(atac, use_rep="X_lsi_no1", n_neighbors=15, n_pcs=min(30, atac.obsm["X_lsi_no1"].shape[1]))
    sc.tl.umap(atac, random_state=0)
    sc.tl.leiden(atac, resolution=0.5, key_added="leiden_atac")
    sc.pl.umap(atac, color=["leiden_atac", "total_counts", "n_genes_by_counts"])


In [ ]:
# OPTIONAL AND POTENTIALLY EXPENSIVE — uncomment only after checking fragments and reference compatibility.
# if atac is not None and atac.uns.get("files", {}).get("fragments"):
#     mu.atac.tl.nucleosome_signal(atac, n=200_000)
#     sc.pl.violin(atac, ["nucleosome_signal"], jitter=0.25)
#
#     # For multiome only, compatible RNA gene intervals can support TSS enrichment.
#     if mdata is not None and "rna" in mdata.mod:
#         tss = mu.atac.tl.tss_enrichment(mdata, n_tss=1000)
#         mu.atac.pl.tss_enrichment(tss)


## 5. Save reproducible outputs and reflect

Save only derived objects to a new results directory. Write a methods record beside them: exact package versions, input source, sample/batch metadata source, filters, doublet policy, graph parameters, and the rationale for every non-default choice.

**AI checkpoint F:** ask an assistant to draft a methods-log template and flag missing provenance. You must check it against your actual commands and results.


In [ ]:
# Save derived, reproducible objects. This does not modify Cell Ranger output or raw counts.
provenance = {
    "python": sys.version,
    "platform": platform.platform(),
    "scanpy": sc.__version__,
    "muon": mu.__version__,
    "input_route": INPUT_ROUTE,
    "input_path": str(required),
    "rna_counts_layer_preserved": bool(rna is not None and "counts" in rna.layers),
    "atac_counts_layer_preserved": bool(atac is not None and "counts" in atac.layers),
}
(RESULTS / "software_and_route.json").write_text(json.dumps(provenance, indent=2))

if rna is not None:
    rna.write(RESULTS / "rna_scanpy_qc_processed.h5ad", compression="gzip")
if atac is not None:
    atac.write(RESULTS / "atac_muon_qc_processed.h5ad", compression="gzip")
if mdata is not None and INPUT_ROUTE == "multiome":
    mdata.mod["rna"] = rna
    mdata.mod["atac"] = atac
    mdata.write(RESULTS / "multiome_qc_processed.h5mu")

print("Saved derived outputs under:", RESULTS)
print("Next: create a short QC/methods log before interpreting clusters.")


## Completion checklist

- [ ] I used an approved project location and did not upload restricted data to Colab/AI services.
- [ ] I recorded my assay, reference, chemistry, input path, samples, and batches.
- [ ] I inspected and documented pre-filter RNA and/or ATAC distributions.
- [ ] I preserved raw RNA counts in `rna.layers["counts"]`.
- [ ] I used transparent, study-specific QC rules rather than universal cutoffs.
- [ ] I treated cluster labels and marker ranks as exploratory pending validation.
- [ ] If I used SAVER, it received only post-QC RNA UMI counts and remains separate from raw counts.
- [ ] I processed ATAC with ATAC-specific QC and LSI, never with SAVER.
- [ ] I saved derived objects and a reproducibility/provenance record.
